# Optional Lab - Neurons and Layers
In this lab we will explore the inner workings of neurons/units and layers. In particular, the lab will draw parallels to the models you have mastered in Course 1, the regression/linear model and the logistic model. The lab will introduce Tensorflow and demonstrate how these models are implemented in that framework.
<figure>
   <img src="./images/C2_W1_NeuronsAndLayers.png"  style="width:540px;height:200px;" >
</figure>


## Packages
**Tensorflow and Keras**  
Tensorflow is a machine learning package developed by Google. In 2019, Google integrated Keras into Tensorflow and released Tensorflow 2.0. Keras is a framework developed independently by François Chollet that creates a simple, layer-centric interface to Tensorflow. This course will be using the Keras interface. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras import Sequential
from tensorflow.keras.losses import MeanSquaredError, BinaryCrossentropy
from tensorflow.keras.activations import sigmoid
from lab_utils_common import dlc
from lab_neurons_utils import plt_prob_1d, sigmoidnp, plt_linear, plt_logistic
plt.style.use('./deeplearning.mplstyle')
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

## Neuron without activation - Regression/Linear Model

### DataSet
We'll use an example from Course 1, linear regression on house prices.

In [ ]:
X_train = np.array([[1.0], [2.0]], dtype=np.float32)           #(size in 1000 square feet)
print("X_train: ", X_train)
print("X_train shape: ", X_train.shape)
Y_train = np.array([[300.0], [500.0]], dtype=np.float32)       #(price in 1000s of dollars)
print("Y_train: ", Y_train)
print("Y_train shape: ", Y_train.shape)

fig, ax = plt.subplots(1,1)
ax.scatter(X_train, Y_train, marker='x', c='r', label="Data Points")
ax.legend( fontsize='xx-large')
ax.set_ylabel('Price (in 1000s of dollars)', fontsize='xx-large')
ax.set_xlabel('Size (1000 sqft)', fontsize='xx-large')
plt.show()

### Regression/Linear Model 
The function implemented by a neuron with no activation is the same as in Course 1, linear regression:
$$ f_{\mathbf{w},b}(x^{(i)}) = \mathbf{w}\cdot x^{(i)} + b \tag{1}$$



We can define a layer with one neuron or unit and compare it to the familiar linear regression function.

In [ ]:
linear_layer = tf.keras.layers.Dense(units=1, activation = 'linear', )

Let's examine the weights.

In [ ]:
print("Let's try and get the wights before doing anything with the layer: ", linear_layer.get_weights())
# In TensorFlow/Keras, weights are not instantiated until the layer processes data for the first time. This is called "lazy initialization."

There are no weights as the weights are not yet instantiated. Let's try the model on one example in `X_train`. This will trigger the instantiation of the weights. Note, the input to the layer must be 2-D, so we'll reshape it.

In [ ]:
# Keras doesn't know the input shape yet, so it can't determine how many weights to create. The weights are only initialized when:
# 1. You pass data through the layer

# we need to reshape X_train[0]
print("X_train[0] before reshaping: ", X_train[0])
print("X_train[0] after reshaping: ", X_train[0].reshape(1,1))
a1 = linear_layer(X_train[0].reshape(1,1))
linear_layer.get_weights()
print(a1)
print("Weights of linear layer after passing data through it: ", linear_layer.get_weights())

# or 2. You explicitly build the layer by specifying the input shape
# linear_layer.build(input_shape=(None, 1))  # Explicitly tell Keras the input shape

# This design allows for flexible model building where the layer can automatically figure out the correct weight dimensions 
# based on the actual input data shape, rather than requiring you to specify it upfront.

What exactly is this "shape" that we are talking about?
Shape is a tuple that describes:
    How many dimensions the array has
    The size of each dimension

Examples:

    1D Array (Vector)
    x = np.array([1, 2, 3, 4])
    # Shape: (4,)  → 1-dimensional array with 4 elements

    2D Array (Matrix)
    X_train = np.array([[1.0], [2.0]])
    # Shape: (2, 1)  → 2 rows, 1 column

The result is a tensor (another name for an array) with a shape of (1,1) or one entry.   
Now let's look at the weights and bias. These weights are randomly initialized to small numbers and the bias defaults to being initialized to zero.

In [ ]:
w, b= linear_layer.get_weights()
print(f"w = {w}, b={b}")

A linear regression model (1) with a single input feature will have a single weight and bias. This matches the dimensions of our `linear_layer` above.   

The weights are initialized to random values so let's set them to some known values.

In [ ]:
set_w = np.array([[200]])
set_b = np.array([100])

# set_weights takes a list of numpy arrays
linear_layer.set_weights([set_w, set_b])
print(linear_layer.get_weights())

Let's compare equation (1) to the layer output.

In [ ]:
# Let's compare two ways of computing the same result to demonstrate that a Dense layer is doing exactly what we learned in Course 1 (linear regression).
a1 = linear_layer(X_train[0].reshape(1,1))
print(a1)
alin = np.dot(set_w,X_train[0].reshape(1,1)) + set_b
print(alin)

They produce the same values!
Now, we can use our linear layer to make predictions on our training data.

In [ ]:
prediction_tf = linear_layer(X_train)
prediction_np = np.dot( X_train, set_w) + set_b

In [ ]:
plt_linear(X_train, Y_train, prediction_tf, prediction_np)

## Neuron with Sigmoid activation
The function implemented by a neuron/unit with a sigmoid activation is the same as in Course 1, logistic  regression:
$$ f_{\mathbf{w},b}(x^{(i)}) = g(\mathbf{w}x^{(i)} + b) \tag{2}$$
where $$g(x) = sigmoid(x)$$ 

Let's set $w$ and $b$ to some known values and check the model.


### DataSet
We'll use an example from Course 1, logistic regression.

In [ ]:
# We already specified the data type as floating point numbers, so the notation "<number>." is redundant here, just a matter of style choice
X_train = np.array([0., 1, 2, 3, 4, 5], dtype=np.float32).reshape(-1,1)  # 2-D Matrix
Y_train = np.array([0,  0, 0, 1, 1, 1], dtype=np.float32).reshape(-1,1)  # 2-D Matrix

In [ ]:
# This code creates boolean masks (arrays of True/False values) to identify which samples belong to which class.
pos = Y_train == 1 # Where are the positive examples (y=1)?
neg = Y_train == 0 # Where are the negative examples (y=0)?

fig,ax = plt.subplots(1,1,figsize=(4,3))
# These boolean masks are used for filtering/indexing to plot different classes with different markers/colors:
ax.scatter(X_train[pos], Y_train[pos], marker='x', s=80, c = 'red', label="y=1")
ax.scatter(X_train[neg], Y_train[neg], marker='o', s=100, label="y=0", facecolors='none', 
              edgecolors=dlc["dlblue"],lw=3)

ax.set_ylim(-0.08,1.1)
ax.set_ylabel('y', fontsize=12)
ax.set_xlabel('x', fontsize=12)
ax.set_title('one variable plot')
ax.legend(fontsize=12)
plt.show()

### Logistic Neuron
We can implement a 'logistic neuron' by adding a sigmoid activation. The function of the neuron is then described by (2) above.   
This section will create a Tensorflow Model that contains our logistic layer to demonstrate an alternate method of creating models. Tensorflow is most often used to create multi-layer models. The [Sequential](https://keras.io/guides/sequential_model/) model is a convenient means of constructing these models.

In [ ]:
model = Sequential(
    [
        # First element is nr of units
        # The model immediately knows the input shape, so it builds the weights right away
        tf.keras.layers.Dense(1, input_dim=1,  activation = 'sigmoid', name='L1')
    ]
)

# We can print the wights without passing data through it because we already specified the input dimension
print(model.get_weights())

# We also get this warning: "Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead."
# This is a deprecation warning in newer versions of TensorFlow/Keras. The API is evolving toward a more explicit and consistent way of defining model inputs.
# Recommended way
# model = Sequential([
#     Input(shape=(1,)),
#     Dense(1, activation='sigmoid')
# ])
# Having a dedicated Input layer makes it crystal clear what shape your model expects

`model.summary()` shows the layers and number of parameters in the model. There is only one layer in this model and that layer has only one unit. The unit has two parameters, $w$ and $b$.

In [ ]:
model.summary()

In [ ]:
logistic_layer = model.get_layer('L1')
w,b = logistic_layer.get_weights()
print(w,b)
print(w.shape,b.shape)

Let's set the weight and bias to some known values.

In [ ]:
set_w = np.array([[2]])
set_b = np.array([-4.5])
# set_weights takes a list of numpy arrays
logistic_layer.set_weights([set_w, set_b])
print(logistic_layer.get_weights())

Let's compare equation (2) to the layer output.

In [ ]:
a1 = model.predict(X_train[0].reshape(1,1))
print(a1)
alog = sigmoidnp(np.dot(set_w,X_train[0].reshape(1,1)) + set_b)
print(alog)

They produce the same values!
Now, we can use our logistic layer and NumPy model to make predictions on our training data.

In [ ]:
plt_logistic(X_train, Y_train, model, set_w, set_b, pos, neg)

The shading above reflects the output of the sigmoid which varies from 0 to 1.

# Congratulations!
You built a very simple neural network and have explored the similarities of a neuron to the linear and logistic regression from Course 1.